In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

LITE_MODE = True
username = "allanhadoop"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [3]:
# Prepare our documents/product summary and prices
y = np.array([float(item.price) for item in train])  # "Go through every item in train, take its price, convert it to a number (float), and put all prices into a NumPy array."
documents = [item.summary for item in train]         # Go through every item in train and take its summary.

In [4]:
# converting text into numbers so the ML model can understand it.
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [5]:
# Define the simple neural network - here is Pytorch code to create a 8 layer neural network
# creates the neural network that will learn to predict the price from the text features. Series of 8 processing steps/layers.
# ReLU - it helps the neural network learn non-linear relationships.
# The __init__ defines the architecture (what layers the network has), while forward() defines how the data travels through those layers.
# 128 and 64 = number of neurons in those layers. They are adjustable design choices, not fixed rules. = Hyper parameter tuning 
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [6]:
# This code takes your text/price data, converts it into PyTorch format, splits it into training and validation data, 
# organizes the training data into batches of 64, and finally creates the neural network ready for training.
"""
                 YOUR DATA
                    ↓
          Text → 5,000 numbers
                    ↓
             PyTorch Tensor FOrmat
                    ↓
             Split the data
              ↙          ↘
          99%             1%
        Training        Validation
           ↓
       Groups of 64
           ↓
     Neural Network
           ↓
    Predicted Price
"""
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [7]:
# This code simply counts how many numbers (parameters) the neural network has to learn.
# neural network has many weights and biases: These weights and biases are called parameters.
""" 
Input
  ↓
[weights + biases]  ← model learns these
  ↓
Layer
  ↓
[weights + biases]  ← model learns these
  ↓
Layer
  ↓
...
  ↓
Prediction
"""
# Trainable parameters = the numbers inside the neural network that the model learns/adjusts during training.

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [11]:
# This is the actual training part of your neural network.
# Give the model data → let it predict → check how wrong it is → adjust the model → repeat.
"""
1. Define how to measure mistakes  MSELoss = Mean Squared Error. How far is the predicted price from the actual price?
2. Create the optimizer. Change the model's weights to reduce the error.Adam is the algorithm being used to make those adjustments.lr=0.001 means learning rate.
3. Train for 2 epochs and Start training . model.train()
4. Take 64 examples at a time - Batch and then 4 key learning steps - Forward pass. Loss. Backward Pass and Optimization 
5. After training - Check the model on validation data model.eval()
              TRAINING
                  ↓
          Take 64 examples
                  ↓
           FORWARD PASS
                  ↓
        Model predicts prices
                  ↓
         Calculate LOSS
                  ↓
       How wrong was the model?
                  ↓
        BACKWARD PASS
                  ↓
    Calculate how weights should change
                  ↓
        OPTIMIZER STEP
                  ↓
       Adjust the weights
                  ↓
          Next 64 examples
                  ↓
                ...
                  ↓
             Epoch 1 done
                  ↓
             Epoch 2 starts
"""
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# We will do 2 complete runs through the data

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/3], Train Loss: 14637.712, Val Loss: 17202.832


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/3], Train Loss: 3026.273, Val Loss: 17479.428


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [3/3], Train Loss: 13871.112, Val Loss: 17327.457


In [12]:
def neural_network(item):
    model.eval()
    with torch.no_grad():                              #"Don't calculate learning/gradient information. We're only predicting. saves memory and computation.
        vector = vectorizer.transform([item.summary])  #it uses the same vectorizer that was used during training.
        vector = torch.FloatTensor(vector.toarray())   #Convert those numbers into PyTorch format
        result = model(vector)[0].item()               #.item() simply extracts the Python number from the PyTorch result.
    return max(0, result)                              # Make sure price isn't negative

In [ ]:
evaluate(neural_network, test)
#Now the error has reduced to $70.73 after 3 epoch

  0%|          | 0/200 [00:00<?, ?it/s]

$128 $19 $28 $14 $6 $178 $53 $45 $2 $172 $257 $171 $33 $292 $7 $7 $30 $20 $133 $58 $79 $9 $94 $27 $220 $306 $355 $22 $7 $53 $16 $103 $92 $7 $60 $74 $51 $41 $102 $37 $172 $23 $9 $43 $137 $42 $29 $24 $21 $28 $10 $73 $110 $56 $131 $15 $20 $84 $12 $27 $110 $27 $57 $27 $448 $87 $52 $288 $3 $284 $15 $12 $153 $36 $26 $63 $46 $27 $19 $17 $86 $138 $1 $54 $2 $60 $8 $138 $84 $108 $20 $15 $2 $9 $21 $71 $2 $35 $153 $279 $29 $64 $7 $95 $4 $16 $50 $324 $5 $40 $42 $113 $118 $21 $93 $147 $197 $31 $31 $24 $7 $349 $35 $21 $106 $24 $7 $130 $37 $103 $8 $81 $71 $2 $56 $47 $118 $28 $54 $58 $34 $110 $33 $250 $145 $91 $26 $370 $154 $4 $1 $117 $8 $52 $3 $127 $177 $20 $70 $17 $126 $5 $8 $15 $260 $4 $105 $33 $6 $15 $7 $10 $169 $4 $17 $27 $10 $45 $43 $31 $313 $8 $218 $53 $1 $38 $78 $6 $17 $3 $107 $15 $17 $49 $13 $52 $39 $1 $21 $0 